In [ ]:
!pip install wikipedia wikipedia-api

In [ ]:
import json

with open("dataset_stage1_base_symptoms.json", "r") as f:
    base_data = json.load(f)

len(base_data)

In [ ]:
import wikipediaapi
import wikipedia

wiki_api = wikipediaapi.Wikipedia('en')

def get_wikipedia_content(disease_name):
    # Try 1: wikipedia-api
    page = wiki_api.page(disease_name)

    if page.exists():
        summary = page.summary

        # try grabbing sections like cause, treatment, diagnosis
        cause = ""
        treatment = ""

        for section in page.sections:
            title = section.title.lower()

            if "cause" in title or "etiology" in title:
                cause = section.text
            if "treatment" in title or "management" in title:
                treatment = section.text

        return summary, cause, treatment

    # Try 2: fallback to wikipedia (less structured)
    try:
        summary = wikipedia.summary(disease_name)
        return summary, "", ""
    except:
        return "", "", ""

In [ ]:
augmented_data = []

for entry in base_data:
    disease = entry["disease"]
    print("Fetching:", disease)

    summary, cause, treatment = get_wikipedia_content(disease)

    # Add to existing entry
    entry["summary"] = summary[:500]  # limit long text
    entry["cause"] = cause[:500] if cause else ""
    entry["treatments"] = [treatment] if treatment else []

    entry["sources"].append("wikipedia")

    augmented_data.append(entry)

In [ ]:
with open("dataset_stage2_wikipedia_augmented.json", "w") as f:
    json.dump(augmented_data, f, indent=4)